In [3]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
#Cell 1: Import Libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
 
from imblearn.over_sampling import SMOTE

In [5]:
#Cell 2: Load Dataset
df = pd.read_csv("creditcard.csv")
 
print(df.head())
print(df.info())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [6]:
#Cell 3: Select Features
X = df.drop("Class", axis=1)
 
y = df["Class"]

In [7]:
#Cell 4: Normalize
scaler = StandardScaler()
 
X = scaler.fit_transform(X)

In [8]:
#Cell 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
 
    X,
    y,
 
    test_size=0.2,
 
    random_state=42,
 
    stratify=y
)

In [9]:
#Cell 6: Convert to Tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
 
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [11]:
#Task 1: Logistic Regression (Weighted Loss)
positive = (y_train==1).sum().item()
 
negative = (y_train==0).sum().item()
 
pos_weight = torch.tensor([negative/positive])
 
model = nn.Linear(30,1)
 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
 
optimizer = optim.Adam(model.parameters(),lr=0.001)
#Training
for epoch in range(50):
 
    output = model(X_train)
 
    loss = criterion(output,y_train)
 
    optimizer.zero_grad()
 
    loss.backward()
 
    optimizer.step()
 
print("Training Complete")

Training Complete


In [12]:
#Evaluation
with torch.no_grad():
 
    pred = torch.sigmoid(model(X_test))
 
    pred_class = (pred>0.5).float()
 
print("Precision:",precision_score(y_test,pred_class))
 
print("Recall:",recall_score(y_test,pred_class))
 
print("F1:",f1_score(y_test,pred_class))
 
print("PR-AUC:",average_precision_score(y_test,pred))

Precision: 0.0026764414964798974
Recall: 0.9387755102040817
F1: 0.005337665351589696
PR-AUC: 0.4322714028735622


In [15]:
#Task 2: MLP
model = nn.Sequential(
 
    nn.Linear(30,64),
 
    nn.ReLU(),
 
    nn.Linear(64,32),
 
    nn.ReLU(),
 
    nn.Linear(32,1)
 
)
 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
 
optimizer = optim.Adam(model.parameters(),lr=0.001)

#Training
for epoch in range(50):
 
    output=model(X_train)
 
    loss=criterion(output,y_train)
 
    optimizer.zero_grad()
 
    loss.backward()
 
    optimizer.step()
 
print("MLP Training Complete")

MLP Training Complete


In [16]:
#Evaluation
with torch.no_grad():
 
    pred=torch.sigmoid(model(X_test))
 
    pred_class=(pred>0.5).float()
 
print("Precision:",precision_score(y_test,pred_class))
 
print("Recall:",recall_score(y_test,pred_class))
 
print("F1:",f1_score(y_test,pred_class))
 
print("PR-AUC:",average_precision_score(y_test,pred))

Precision: 0.1332312404287902
Recall: 0.8877551020408163
F1: 0.23169107856191745
PR-AUC: 0.7225635567193437


In [17]:
#Task 4: SMOTE
smote = SMOTE(random_state=42)
 
X_smote,y_smote = smote.fit_resample(X,y)
 
print(X_smote.shape)
 
print(y_smote.value_counts())

(568630, 30)
Class
0    284315
1    284315
Name: count, dtype: int64
